# Cross-Encoder Reranking & Pairwise Preference Scores

Model: `cross-encoder/ms-marco-MiniLM-L-6-v2`  

### What this notebook does
1. Loads queries, BM25 candidates, and qrels from your existing files
2. Scores each (query, passage) pair with the cross-encoder -> pointwise scores `s(q, d)`
3. Reranks the candidate list per query by descending score
4. Constructs pairwise preference scores `g(q, di, dj) = s(q, di) - s(q, dj)` for relevant vs. non-relevant pairs
5. Saves both outputs to disk for downstream IG attribution

Notes:
- The cross-encoder is trained with a pointwise binary-relevance objective
- Pairwise preferences are derived post-hoc via score differences, consistent with the formulation in §3 of the proposal

In [22]:
# -- IMPORTS --
import json
import torch
import pickle
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import CrossEncoder

In [12]:
queries_file = "../data/msmarco_passage_dev/raw/queries.dev.tsv"
candidates_file = "../data/msmarco_passage_dev/raw/top1000.dev"
qrels_file = "../data/msmarco_passage_dev/raw/qrels.dev.tsv"

out_dir = Path("../outputs")
out_dir.mkdir(exist_ok=True)

model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
top_k = 100
batch_size = 64

# for pairwise pairs: how many (relevant, non-relevant) pairs to keep per query.
max_pairs_per_query = 5

# set seed for reproducibility
seed = 42
np.random.seed(seed)

## Load Data

In [24]:
queries = pd.read_csv(
    queries_file, sep="\t", header=None, names=["qid", "query"])

queries["qid"] = queries["qid"].astype(str)
query_map = dict(zip(queries["qid"], queries["query"]))

# for a pilot, we can limit the number of queries
candidate_qids = set(pd.read_csv(candidates_file, sep="\t", header=None, usecols=[0], names=["qid"])["qid"].astype(str))
query_map = {qid: q for qid, q in query_map.items() if qid in candidate_qids}
query_map = dict(list(query_map.items())[:100])
print(f"Pilot queries with candidates: {len(query_map)}")

print(f"Queries loaded: {len(query_map):,}")

Pilot queries with candidates: 100
Queries loaded: 100


In [25]:
# bm25 candidates 
candidates_df = pd.read_csv(
    candidates_file, sep="\t", header=None,
    names=["qid", "pid", "query", "passage"])

candidates_df["qid"] = candidates_df["qid"].astype(str)
candidates_df["pid"] = candidates_df["pid"].astype(str)

# Keep only queries that appear in our query set, and only top-K candidates
candidates_df = candidates_df[candidates_df["qid"].isin(query_map)]
candidates_df = (candidates_df.groupby("qid", sort=False).head(top_k).reset_index(drop=True))

print(f"Candidate rows (after top_k={top_k} filter): {len(candidates_df):,}")

Candidate rows (after top_k=100 filter): 9,339


In [20]:
qrels_df = pd.read_csv(
    qrels_file, sep="\t", header=None,
    names=["qid", "_", "pid", "relevance"])

qrels_df["qid"] = qrels_df["qid"].astype(str)
qrels_df["pid"] = qrels_df["pid"].astype(str)

qrel_map = {(row.qid, row.pid): row.relevance
    for row in qrels_df.itertuples()}

print(f"Qrel entries: {len(qrel_map):,}")

Qrel entries: 59,273


## Load Cross-Encoder

In [26]:
model = CrossEncoder(model_name, max_length=512)
print(f"Loaded: {model_name}")
print(f"Device: {model.model.device}")

Loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
Device: mps:0


In [27]:
if torch.backends.mps.is_available():
    model.model = model.model.to("mps")
    print("Using MPS (Apple Silicon)")
else:
    print("MPS not available, using CPU")

Using MPS (Apple Silicon)


## Score & Rerank

For each query, score all top_k (query, passage) pairs and sort by descending score.  
This gives us `s(q, d)` - the pointwise relevance logit from the cross-encoder.

In [28]:
ranked_results = {}

for qid, group in tqdm(candidates_df.groupby("qid"), desc="Reranking queries"):
    query_text = query_map[qid]
    passages = group["passage"].tolist()
    pids = group["pid"].tolist()

    # score all (query, passage) pairs in one batched call
    pairs  = [[query_text, p] for p in passages]
    scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=False)

    # sort by score descending
    order = np.argsort(scores)[::-1]

    ranked_results[qid] = [
        {"pid": pids[i],
        "passage": passages[i],
        "score": float(scores[i]),
        "rank": rank + 1,
        "relevant": int(qrel_map.get((qid, pids[i]), 0) > 0),}
        for rank, i in enumerate(order)]

print(f"\nReranked {len(ranked_results):,} queries.")

Reranking queries:   0%|          | 0/100 [00:00<?, ?it/s]


Reranked 100 queries.


In [30]:
all_scores = [entry["score"] for results in ranked_results.values() for entry in results]
print(f"Score range: [{min(all_scores):.3f}, {max(all_scores):.3f}]")
print(f"Score mean: {np.mean(all_scores):.3f}")
print(f"Score median: {np.median(all_scores):.3f}")

Score range: [-11.453, 11.398]
Score mean: -5.220
Score median: -6.368


In [31]:
rel_ranks, nonrel_ranks = [], []
for results in ranked_results.values():
    for entry in results:
        (rel_ranks if entry["relevant"] else nonrel_ranks).append(entry["rank"])

print(f"Mean rank of RELEVANT passages: {np.mean(rel_ranks):.1f}")
print(f"Mean rank of NON-RELEVANT passages: {np.mean(nonrel_ranks):.1f}")
print("(Lower rank = higher in the list - relevant should be much lower)")

Mean rank of RELEVANT passages: 2.9
Mean rank of NON-RELEVANT passages: 50.0
(Lower rank = higher in the list - relevant should be much lower)


## Pairwise Preference Scores

For each query, pair each relevant passage `d` with each non-relevant passage `dj` and compute the pairwise preference score:

$$g(q, d_i, d_j) = s(q, d_i) - s(q, d_j)$$

If $g > 0$ the model correctly prefers the relevant document.  
These pairs are the inputs to Integrated Gradients in the next phase.

In [ ]:
pairwise_records = []
rng = np.random.default_rng(seed)

for qid, results in ranked_results.items():
    query_text = query_map[qid]

    rel_entries = [e for e in results if e["relevant"]]
    nonrel_entries = [e for e in results if not e["relevant"]]

    # skip queries with no relevant passage in the candidate set
    if not rel_entries or not nonrel_entries:
        continue

    # all (rel, non-rel) combinations
    pairs = [(di, dj) for di in rel_entries for dj in nonrel_entries]

    # cap if needed
    if max_pairs_per_query and len(pairs) > max_pairs_per_query:
        idx = rng.choice(len(pairs), size=max_pairs_per_query, replace=False)
        pairs = [pairs[i] for i in idx]

    for di, dj in pairs:
        g = di["score"] - dj["score"]   # g(q, di, dj)
        pairwise_records.append({
            "qid": qid,
            "query": query_text,
            "pid_i": di["pid"],
            "passage_i": di["passage"],
            "score_i": di["score"],
            "pid_j": dj["pid"],
            "passage_j": dj["passage"],
            "score_j": dj["score"],
            "g_score": g,
            "correct_pref": int(g > 0)})

pairs_df = pd.DataFrame(pairwise_records)
print(f"Total pairwise pairs: {len(pairs_df):,}")
print(f"Queries covered: {pairs_df['qid'].nunique():,}")
print(f"\nModel preference accuracy (g > 0): "f"{pairs_df['correct_pref'].mean():.1%}")

Total pairwise pairs: 90
Queries covered: 18

Model preference accuracy (g > 0): 96.7%


In [ ]:
# distribution of g scores
print("g-score distribution:")
print(pairs_df["g_score"].describe().round(3))
print(f"\nPairs where model CORRECTLY prefers relevant doc (g>0): "
      f"{pairs_df['correct_pref'].sum():,} / {len(pairs_df):,}")
print(f"Pairs where model INCORRECTLY prefers non-relevant (g≤0): "
      f"{(pairs_df['correct_pref']==0).sum():,} / {len(pairs_df):,}")

g-score distribution:
count    90.000
mean     10.543
std       6.595
min      -1.244
25%       6.045
50%      10.113
75%      16.540
max      21.124
Name: g_score, dtype: float64

Pairs where model CORRECTLY prefers relevant doc (g>0): 87 / 90
Pairs where model INCORRECTLY prefers non-relevant (g≤0): 3 / 90


## Save Outputs

In [ ]:
ranked_out = out_dir / "ranked_results.pkl"
with open(ranked_out, "wb") as f: pickle.dump(ranked_results, f)
print(f"Saved ranked results → {ranked_out}")

ranked_flat = out_dir / "ranked_results.csv"
rows = [{"qid": qid, **entry} for qid, results in ranked_results.items() for entry in results]

pd.DataFrame(rows).to_csv(ranked_flat, index=False)
print(f"Saved ranked results (flat CSV) → {ranked_flat}")

Saved ranked results → ../outputs/ranked_results.pkl
Saved ranked results (flat CSV) → ../outputs/ranked_results.csv


In [35]:
pairs_pkl = out_dir / "pairwise_scores.pkl"
pairs_csv = out_dir / "pairwise_scores.csv"

pairs_df.to_pickle(pairs_pkl)
pairs_df.to_csv(pairs_csv, index=False)

print(f"Saved pairwise scores (pkl) → {pairs_pkl}")
print(f"Saved pairwise scores (csv) → {pairs_csv}")

Saved pairwise scores (pkl) → ../outputs/pairwise_scores.pkl
Saved pairwise scores (csv) → ../outputs/pairwise_scores.csv
